# INFO-H-515 Project: Task 3 - Performance Evaluation Suite
**Team:** 7

**Objective:** Implementing three cumulative performance options:
1. **Option 2:** Quality & retrieval comparison between embedding strategies (SBERT vs. GloVe vs. TF-IDF).
2. **Option 3:** Algorithmic and latency comparison between distance metrics (Cosine vs. Euclidean).
3. **Option 4:** Distributed Approximate Nearest Neighbor search using **Locality-Sensitive Hashing (LSH)**.

In [4]:
# Import necessary libraries
import json
import re
import math
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col
from pyspark.sql.types import DoubleType, ArrayType
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml.feature import BucketedRandomProjectionLSH
from sentence_transformers import SentenceTransformer
from pyspark.ml.feature import Tokenizer, HashingTF

In [5]:
#this is for building the ground truth for evaluation
# its just to check if the chunks are correctly extracted and embedded, we will use it to check if the retrieved chunks are relevant to the question asked by the user
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Recherche_GroundTruth") \
    .master("local[*]") \
    .getOrCreate()
df_chunks = spark.read.parquet("data/data_processed/embedded_chunks.parquet")

# Example: Show chunks from a specific PDF and page
df_chunks.filter(
    (col("source_pdf") == "5-nosql.pdf") & 
    (col("page_num") == 20)
).select("chunk_id", "chunk_text").show(truncate=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 12:26:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
# Subtask 3.1: Initialization & Ground Truth Setup
# 1. Initialize the Spark Context for local parallel execution
spark = SparkSession.builder \
    .appName("BigData_RAG_Task3_Complete_Evaluation") \
    .getOrCreate()

# 2. Ingest the multi-strategy data schema produced in Task 1
parquet_dir = "data/data_processed/embedded_chunks.parquet"
df_chunks = spark.read.parquet(parquet_dir)

# 3. Load the sentence transformer model for query processing
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 4. Define the Gold Standard Ground Truth Evaluation Set
evaluation_set = [
    {
        "query": "Explain the key differences between the Google File System (GFS) and the Hadoop Distributed File System (HDFS).",
        "expected_chunk_ids": ["1-intro.pdf_p50_c1", "1-intro.pdf_p48_c1"],
    },
    {
        "query": "What is the concept of Time series forecasting?",
        "expected_chunk_ids": [
            "a_the-big-book-of-machine-learning-use-cases.pdf_p25_c1", 
            "a_the-big-book-of-machine-learning-use-cases.pdf_p18_c1", 
            "a_the-big-book-of-machine-learning-use-cases.pdf_p20_c1"
        ],
    },
    {
        "query": "What is the main difference between NoSQL and traditional relational databases?",
        "expected_chunk_ids": [
            "5-nosql.pdf_p20_c2", 
            "5-nosql.pdf_p13_c1", 
            "5-nosql.pdf_p14_c1"
        ],
    }
]

print(f"Evaluation pipeline ready with {len(evaluation_set)} validation queries.")

26/05/21 12:26:45 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
/home/anischaibi/mon_env/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluation pipeline ready with 3 validation queries.


In [3]:
# Task 3.2: Option 2 (Embedding Comparison) & Option 3 (Distance Metric Comparison)

def run_retrieval_benchmark(query_text, expected_chunks, df, embedding_col, metric_type="cosine", k=5):
    """
    Executes an exact retrieval sweep over the cluster to assess quality and latency 
    across different embedding strategies (SBERT, GloVe, TF-IDF).
    """
    # 1. Encode the query based on the active embedding strategy
    if "glove" in embedding_col:
        # Local fallback representation for GloVe averaging
        import gensim.downloader as api
        try: glove_model = api.load("glove-wiki-gigaword-100")
        except: glove_model = {}
        words = query_text.lower().split()
        vectors = [glove_model[w] for w in words if w in glove_model]
        query_vec = [float(x) for x in (sum(vectors)/len(vectors)).tolist()] if vectors else [0.0]*100
        
    elif "tfidf" in embedding_col:
        # Reconstruct native PySpark HashingTF pipeline for the query
        query_df = spark.createDataFrame([(query_text,)], ["chunk_text"])
        tokenizer = Tokenizer(inputCol="chunk_text", outputCol="words")
        # Ensure numFeatures matches the Task 1 configuration (384)
        htf = HashingTF(inputCol="words", outputCol="raw_features", numFeatures=384)
        
        words_df = tokenizer.transform(query_df)
        tf_df = htf.transform(words_df)
        
        # Extract the SparseVector and convert to a dense Python list
        sparse_vec = tf_df.select("raw_features").first()[0]
        query_vec = sparse_vec.toArray().tolist()
        
    else:
        # Default dense encoding using Sentence Transformers (SBERT)
        query_vec = embedder.encode(query_text).tolist()
        
    
    # 2. Define dynamic Distance UDFs

    if metric_type == "cosine":
        def compute_cosine(chunk_vec):
            if not chunk_vec or not query_vec: return 0.0
            dot_prod = sum(a * b for a, b in zip(query_vec, chunk_vec))
            norm_a = math.sqrt(sum(a * a for a in query_vec))
            norm_b = math.sqrt(sum(b * b for b in chunk_vec))
            return dot_prod / (norm_a * norm_b) if norm_a and norm_b else 0.0
        distance_udf = udf(compute_cosine, DoubleType())
        sort_ascending = False # Higher cosine is better
    else:
        def compute_euclidean(chunk_vec):
            if not chunk_vec or not query_vec: return float('inf')
            return math.sqrt(sum((a - b) ** 2 for a, b in zip(query_vec, chunk_vec)))
        distance_udf = udf(compute_euclidean, DoubleType())
        sort_ascending = True # Lower euclidean distance is better


    # 3. Distributed Scoring and Metric Calculation

    start_time = time.time()
    scored_df = df.withColumn("score", distance_udf(col(embedding_col)))
    
    # Retrieve Top K
    if sort_ascending:
        top_chunks = scored_df.orderBy(col("score").asc()).limit(k).collect()
    else:
        top_chunks = scored_df.orderBy(col("score").desc()).limit(k).collect()
        
    latency = time.time() - start_time
    
    # Calculate Precision and Recall against Ground Truth
    retrieved_ids = [row.chunk_id for row in top_chunks]
    hits = len(set(retrieved_ids).intersection(set(expected_chunks)))
    
    precision = hits / k
    recall = hits / len(expected_chunks) if expected_chunks else 0.0
    
    return precision, recall, latency

# Execute cross-matrix analysis on the ENTIRE Evaluation Set
print("--- RUNNING CUMULATIVE BENCHMARK (OPTION 2 & OPTION 3) ---")

# Get the total number of queries in the Ground Truth
num_queries = len(evaluation_set)

# Looping over all 3 strategies configured in Task 1
for emb in ["embedding", "embedding_glove", "embedding_tfidf"]: 
    for metric in ["cosine", "euclidean"]:     
        
        # Initialize accumulators for calculating the mean metrics
        total_precision = 0.0
        total_recall = 0.0
        total_latency = 0.0
        
        # Iterate over ALL test cases in the Ground Truth
        for test_case in evaluation_set:
            p, r, t = run_retrieval_benchmark(
                test_case["query"], 
                test_case["expected_chunk_ids"],
                df_chunks, 
                emb, 
                metric,
                k=5
            )
            total_precision += p
            total_recall += r
            total_latency += t
            
        # Calculate the final averages (Mean Average Precision, Mean Recall, Average Latency)
        avg_precision = total_precision / num_queries
        avg_recall = total_recall / num_queries
        avg_latency = total_latency / num_queries
        
        # Format and display the aggregated results cleanly
        print(f"Strategy: [{emb.ljust(15)}] | Metric: [{metric.ljust(9)}] -> Mean Precision@5: {avg_precision*100:4.1f}% | Mean Recall@5: {avg_recall*100:4.1f}% | Avg Latency: {avg_latency:.4f}s")

--- RUNNING CUMULATIVE BENCHMARK (OPTION 2 & OPTION 3) ---
Strategy: [embedding      ] | Metric: [cosine   ] -> Mean Precision@5: 26.7% | Mean Recall@5: 55.6% | Avg Latency: 0.3135s
Strategy: [embedding      ] | Metric: [euclidean] -> Mean Precision@5: 26.7% | Mean Recall@5: 55.6% | Avg Latency: 0.2344s
Strategy: [embedding_glove] | Metric: [cosine   ] -> Mean Precision@5:  0.0% | Mean Recall@5:  0.0% | Avg Latency: 0.1514s
Strategy: [embedding_glove] | Metric: [euclidean] -> Mean Precision@5:  0.0% | Mean Recall@5:  0.0% | Avg Latency: 0.1492s
Strategy: [embedding_tfidf] | Metric: [cosine   ] -> Mean Precision@5: 13.3% | Mean Recall@5: 27.8% | Avg Latency: 0.2369s
Strategy: [embedding_tfidf] | Metric: [euclidean] -> Mean Precision@5:  6.7% | Mean Recall@5: 16.7% | Avg Latency: 0.2081s


In [7]:

# Task 3.3: Option 4 (Approximate Similarity Search using Native PySpark LSH)
# 1. Transform the standard array column into a PySpark ML native Vector structure
array_to_vector_udf = udf(lambda array_list: Vectors.dense(array_list), VectorUDT())
df_vectorized = df_chunks.withColumn("native_vector_sbert", array_to_vector_udf(col("embedding")))

# 2. Instantiate and fit the Bucketed Random Projection LSH model for Euclidean distance spaces
lsh = BucketedRandomProjectionLSH(
    inputCol="native_vector_sbert", 
    outputCol="hashes", 
    bucketLength=2.5, 
    numHashTables=5
)
lsh_model = lsh.fit(df_vectorized)
df_hashed = lsh_model.transform(df_vectorized)

# 3. Benchmark the Approximate Search Mechanism against the first test query
query_text = evaluation_set[0]["query"]
query_dense_vector = Vectors.dense(embedder.encode(query_text).tolist())

start_lsh = time.time()
# Execute approximate nearest neighbor resolution natively on the cluster state
lsh_results = lsh_model.approxNearestNeighbors(
    df_hashed, 
    query_dense_vector, 
    numNearestNeighbors=5
).collect()
elapsed_lsh = time.time() - start_lsh

print(f"\n[LSH Approximate Retrieval Profile]")
print(f"Approximate Query Latency: {elapsed_lsh:.4f} seconds")
print("Top 3 Chunks Retrieved via Locality-Sensitive Hashing:")
for row in lsh_results[:3]:
    print(f"  - Chunk ID: {row.chunk_id}")


[LSH Approximate Retrieval Profile]
Approximate Query Latency: 1.4655 seconds
Top 3 Chunks Retrieved via Locality-Sensitive Hashing:
  - Chunk ID: info-H505_Group_P7.pdf_p4_c1
  - Chunk ID: info-H505_Group_P7.pdf_p6_c1
  - Chunk ID: info-H505_Group_P7.pdf_p3_c1


In [8]:
# Task 3.4: Format-Adherence and Grounding Metrics (LLM Generation Evaluation)

# 1. Load the generated results from Task 2
file_path = "data/data_processed/generated_qa.json"

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        # If the file contains a single dict, convert to list for iteration
        if isinstance(data, dict):
            data = [data]
except FileNotFoundError:
    print(f"Error: Could not find {file_path}. Please check the path.")
    data = []

total_queries = len(data)
total_questions_expected = total_queries

successfully_parsed_questions = 0
questions_with_valid_citations = 0

# 2. Regular Expression to parse the specific format you requested from the LLM
# Matches: Q: [text] \n A: [text] \n Citation: [text]
qa_pattern = re.compile(r"Q:\s*(.*?)\nA:\s*(.*?)\nCitation:\s*(.*?)(?=\n\nQuestion|\Z)", re.DOTALL)

for item in data:
    llm_output = item.get("llm_output", "")
    num_sources_provided = len(item.get("retrieved_chunks_used", []))
    
    # Extract all questions from the LLM text output
    extracted_questions = qa_pattern.findall(llm_output)
    
    for q, a, citation in extracted_questions:
        successfully_parsed_questions += 1
        
        # Grounding check
        cited_numbers = re.findall(r'\[(\d+)\]', citation)
        
        is_citation_valid = False
        for num_str in cited_numbers:
            num = int(num_str)
            # Check if the cited source number actually exists in the provided context
            if 1 <= num <= num_sources_provided:
                is_citation_valid = True
                
        if is_citation_valid:
            questions_with_valid_citations += 1

# 3. Calculate Final Metrics
if total_questions_expected > 0:
    format_adherence_rate = (successfully_parsed_questions / total_questions_expected) * 100
else:
    format_adherence_rate = 0.0

if successfully_parsed_questions > 0:
    citation_validity_rate = (questions_with_valid_citations / successfully_parsed_questions) * 100
else:
    citation_validity_rate = 0.0

print(f"Total Queries Processed: {total_queries}")
print(f"Expected Questions: {total_questions_expected} | Parsed Successfully: {successfully_parsed_questions}")
print("-" * 50)
print(f"Format-Adherence (Parser Pass Rate): {format_adherence_rate:.1f}%")
print(f"Citation Grounding (Valid Citation Rate): {citation_validity_rate:.1f}%")

Total Queries Processed: 1
Expected Questions: 1 | Parsed Successfully: 3
--------------------------------------------------
Format-Adherence (Parser Pass Rate): 300.0%
Citation Grounding (Valid Citation Rate): 100.0%
